# 02 - LangChain y chunking

Objetivo: cargar documentos y dividirlos en chunks razonables antes de vectorizar.


In [1]:
import importlib.util
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from markitdown import MarkItDown

load_dotenv(Path("..").resolve() / ".env")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))
DOCS_DIR = Path("..").resolve() / "docs"

print({
    "DOCS_DIR": str(DOCS_DIR),
    "CHUNK_SIZE": CHUNK_SIZE,
    "CHUNK_OVERLAP": CHUNK_OVERLAP,
})


{'DOCS_DIR': 'C:\\Repos\\rag-local-lab\\docs', 'CHUNK_SIZE': 650, 'CHUNK_OVERLAP': 100}


## Cargar Markdown ya preparado

Empezamos por el caso mas sencillo: documentos que ya estan en `.md` dentro de `docs/`.


In [2]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()
print(f"Documentos cargados: {len(documents)}")
documents[0]


Documentos cargados: 2


Document(metadata={'source': 'C:\\Repos\\rag-local-lab\\docs\\manual_mesa_oslo.md'}, page_content='# Manual de montaje — Mesa Oslo\n\n**Producto:** Mesa Oslo  \n**Referencia interna:** FH-MOSLO-140\n\n## 1. Contenido de la caja\nAntes de empezar, verifica que el embalaje contiene:\n- 1 tablero principal,\n- 4 patas,\n- 2 travesaños,\n- 12 tornillos largos,\n- 4 tornillos cortos,\n- 1 llave Allen incluida.\n\n## 2. Herramientas recomendadas\nPara el montaje se recomienda:\n- la **llave Allen incluida**,\n- un **destornillador Phillips**,\n- un **mazo de goma opcional** para ajustes suaves,\n- **dos personas** para manipular el tablero sin dañarlo.\n\n## 3. Preparación previa\nMonta la mesa sobre una superficie limpia y protegida. No arrastres el tablero sobre suelos rugosos. Se recomienda dejar todas las piezas presentadas antes de apretar por completo la tornillería.\n\n## 4. Secuencia de montaje\n1. Coloca el tablero boca abajo sobre una manta o cartón protector.\n2. Presenta las 4 pa

## Convertir otros formatos a Markdown con `markitdown`

Cuando los documentos de origen no vienen en Markdown, una opcion muy comoda es convertirlos antes a texto estructurado. `markitdown` sirve para formatos como `pdf`, `docx`, `pptx`, `xlsx` o `html`.

La idea es sencilla:

1. localizar ficheros de otros tipos,
2. convertir cada fichero a Markdown,
3. envolver el resultado en `Document` de LangChain,
4. chunkear todo igual que si hubiera nacido en `.md`.


In [3]:
markitdown = MarkItDown()
other_source_extensions = [".pdf", ".docx", ".pptx", ".xlsx", ".html"]

other_source_files = [
    path
    for path in DOCS_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in other_source_extensions
]

missing_optional_dependencies = []
if any(path.suffix.lower() == ".docx" for path in other_source_files):
    if importlib.util.find_spec("mammoth") is None:
        missing_optional_dependencies.append("DOCX requiere instalar `markitdown[docx]`.")

if missing_optional_dependencies:
    raise RuntimeError(
        "Faltan dependencias opcionales de MarkItDown: "
        + " ".join(missing_optional_dependencies)
        + " Ejecuta `pip install -r requirements-local.txt` y reinicia el kernel."
    )

converted_documents = []
for path in other_source_files:
    result = markitdown.convert(str(path))
    converted_documents.append(
        Document(
            page_content=result.text_content,
            metadata={
                "source": str(path),
                "converted_with": "markitdown",
                "original_extension": path.suffix.lower(),
            },
        )
    )

print({
    "ficheros_encontrados": len(other_source_files),
    "documentos_convertidos": len(converted_documents),
})

if converted_documents:
    converted_documents[0]
else:
    print("No hay todavia archivos PDF, DOCX, PPTX, XLSX o HTML dentro de docs/.")


{'ficheros_encontrados': 1, 'documentos_convertidos': 1}


In [4]:
all_documents = documents + converted_documents
print(f"Documentos totales para chunking: {len(all_documents)}")


Documentos totales para chunking: 3


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(all_documents)
print(f"Chunks generados: {len(chunks)}")

for idx, chunk in enumerate(chunks[:3]):
    chunk.metadata["chunk_id"] = idx
    print("=" * 80)
    print(chunk.metadata.get("source", "sin source"))
    print(chunk.page_content[:400])


Chunks generados: 12
C:\Repos\rag-local-lab\docs\manual_mesa_oslo.md
# Manual de montaje — Mesa Oslo

**Producto:** Mesa Oslo  
**Referencia interna:** FH-MOSLO-140

## 1. Contenido de la caja
Antes de empezar, verifica que el embalaje contiene:
- 1 tablero principal,
- 4 patas,
- 2 travesaños,
- 12 tornillos largos,
- 4 tornillos cortos,
- 1 llave Allen incluida.

## 2. Herramientas recomendadas
Para el montaje se recomienda:
- la **llave Allen incluida**,
- un **
C:\Repos\rag-local-lab\docs\manual_mesa_oslo.md
## 3. Preparación previa
Monta la mesa sobre una superficie limpia y protegida. No arrastres el tablero sobre suelos rugosos. Se recomienda dejar todas las piezas presentadas antes de apretar por completo la tornillería.

## 4. Secuencia de montaje
1. Coloca el tablero boca abajo sobre una manta o cartón protector.
2. Presenta las 4 patas en sus posiciones.
3. Une los travesaños a las patas sin ap
C:\Repos\rag-local-lab\docs\manual_mesa_oslo.md
## 5. Importante
**No aprietes to

In [6]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]
print({
    "min": min(chunk_lengths),
    "max": max(chunk_lengths),
    "media_aprox": round(sum(chunk_lengths) / len(chunk_lengths), 1),
})


{'min': 312, 'max': 633, 'media_aprox': 479.2}


Prueba sugerida:

1. Cambia `CHUNK_SIZE` o `CHUNK_OVERLAP` en `.env`.
2. Vuelve a ejecutar este notebook.
3. Observa si cambian los trozos que luego irian a embeddings.
